# A100 Qwen2.5-1.5B Phishing Detection Fine-tuning

只需要修改下一個 cell 的 `ZIP_PATH`，即可從解壓縮資料開始微調。

預設方法：LoRA BF16 supervised fine-tuning。

資料格式需求：zip 內包含 `train.csv`、`valid.csv`、`test.csv`，欄位為 `subject`、`body`、`label`、`dataset`。

In [ ]:
# 只需要修改這個路徑
ZIP_PATH = "/content/dataset.zip"

# 可依 A100 40GB / 80GB 調整，但預設值已可直接訓練
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./qwen25_1_5b_phishing_lora_a100"
EPOCHS = 3
MAX_LENGTH = 2048
BATCH_SIZE = 8
GRAD_ACCUM = 2
LEARNING_RATE = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
SEED = 42

# 若 A100 80GB 且想保留更長 email，可改為 4096。
# MAX_LENGTH = 4096

In [ ]:
# Some A100 notebook images preinstall torchao==0.10.0, which breaks recent PEFT.
# LoRA BF16 does not require torchao, so removing it is the most stable fix.
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.45" datasets peft accelerate trl scikit-learn pandas

In [ ]:
import os
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

## 1. 解壓縮資料集

In [ ]:
zip_path = Path(ZIP_PATH)
assert zip_path.exists(), f'ZIP_PATH not found: {zip_path}'

DATA_DIR = Path('./dataset_unzipped')
DATA_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(DATA_DIR)

print('Extracted files:')
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_file():
        print('-', p, p.stat().st_size)

## 2. 讀取與檢查 train / valid / test

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
valid_df = pd.read_csv(DATA_DIR / 'valid.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

required_cols = {'subject', 'body', 'label', 'dataset'}
for name, df in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    missing = required_cols - set(df.columns)
    assert not missing, f'{name} missing columns: {missing}'
    print(name, df.shape)
    print(df['label'].value_counts().sort_index().to_dict())
    print(df.groupby(['dataset', 'label']).size())
    print()

display(train_df.head(3))

## 3. 轉換成 instruction tuning 格式

In [ ]:
SYSTEM_PROMPT = (
    'You are a cybersecurity email classifier. '
    'Decide whether the email is phishing or legitimate. '
    'Return exactly one word only: phishing or legitimate.'
)

def label_to_text(label):
    return 'phishing' if int(label) == 1 else 'legitimate'

def make_email_text(row):
    subject = '' if pd.isna(row.get('subject')) else str(row.get('subject'))
    body = '' if pd.isna(row.get('body')) else str(row.get('body'))
    return f'Subject: {subject}\nBody: {body}'

def make_messages(row, include_answer=True):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': make_email_text(row)},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': label_to_text(row['label'])})
    return messages

train_df['target_text'] = train_df['label'].map(label_to_text)
valid_df['target_text'] = valid_df['label'].map(label_to_text)
test_df['target_text'] = test_df['label'].map(label_to_text)

## 4. 載入 Qwen tokenizer / model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

print('Loaded:', MODEL_NAME)
print('Memory footprint GB:', round(model.get_memory_footprint() / 1024**3, 2))

## 5. Tokenize 並只訓練 assistant answer

In [ ]:
def tokenize_row(row):
    prompt_text = tokenizer.apply_chat_template(
        make_messages(row, include_answer=False),
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        make_messages(row, include_answer=True),
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )['input_ids']
    tokenized = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    labels = tokenized['input_ids'].copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    if all(x == -100 for x in labels):
        labels[-1] = tokenized['input_ids'][-1]
    tokenized['labels'] = labels
    return tokenized

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize_row, remove_columns=list(train_df.columns))
valid_ds = Dataset.from_pandas(valid_df, preserve_index=False).map(tokenize_row, remove_columns=list(valid_df.columns))

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

print(train_ds)
print(valid_ds)

## 6. 掛載 LoRA adapter

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. 開始微調

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    bf16=True,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    report_to='none',
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    gradient_checkpointing=True,
    optim='adamw_torch',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(f'{OUTPUT_DIR}/best_adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/best_adapter')

## 8. 測試集推論

In [ ]:
def parse_prediction(text):
    s = text.lower().strip()
    p = s.find('phishing')
    l = s.find('legitimate')
    if p >= 0 and (l < 0 or p < l):
        return 1
    if l >= 0:
        return 0
    return -1

@torch.inference_mode()
def predict_one(row, max_new_tokens=8):
    prompt_text = tokenizer.apply_chat_template(
        make_messages(row, include_answer=False),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0, inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return raw, parse_prediction(raw)

pred_rows = []
for i, row in test_df.iterrows():
    raw, pred = predict_one(row)
    pred_rows.append({
        'index': i,
        'dataset': row['dataset'],
        'label': int(row['label']),
        'prediction': pred,
        'raw_response': raw,
        'subject': row['subject'],
    })
    if (i + 1) % 100 == 0:
        print(f'predicted {i + 1}/{len(test_df)}')

pred_df = pd.DataFrame(pred_rows)
pred_path = f'{OUTPUT_DIR}/test_predictions.csv'
pred_df.to_csv(pred_path, index=False, encoding='utf-8-sig')
print('saved:', pred_path)
display(pred_df.head())

## 9. 計算 Accuracy / Precision / Recall / F1

In [ ]:
valid_pred_df = pred_df[pred_df['prediction'] != -1].copy()
invalid = len(pred_df) - len(valid_pred_df)

y_true = valid_pred_df['label'].astype(int).to_numpy()
y_pred = valid_pred_df['prediction'].astype(int).to_numpy()

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, pos_label=1, average='binary', zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

overall = pd.DataFrame([{
    'n': len(valid_pred_df),
    'invalid': invalid,
    'TP': tp,
    'TN': tn,
    'FP': fp,
    'FN': fn,
    'Accuracy': acc,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1,
}])
display(overall)

rows = []
for ds, group in valid_pred_df.groupby('dataset'):
    yt = group['label'].astype(int).to_numpy()
    yp = group['prediction'].astype(int).to_numpy()
    ds_acc = accuracy_score(yt, yp)
    ds_precision, ds_recall, ds_f1, _ = precision_recall_fscore_support(yt, yp, pos_label=1, average='binary', zero_division=0)
    ds_tn, ds_fp, ds_fn, ds_tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    rows.append({
        'dataset': ds,
        'n': len(group),
        'TP': ds_tp,
        'TN': ds_tn,
        'FP': ds_fp,
        'FN': ds_fn,
        'Accuracy': ds_acc,
        'Precision': ds_precision,
        'Recall': ds_recall,
        'F1-score': ds_f1,
    })

by_dataset = pd.DataFrame(rows).sort_values('dataset')
display(by_dataset)

overall.to_csv(f'{OUTPUT_DIR}/test_metrics_overall.csv', index=False, encoding='utf-8-sig')
by_dataset.to_csv(f'{OUTPUT_DIR}/test_metrics_by_dataset.csv', index=False, encoding='utf-8-sig')

## 10. 與未微調 baseline 比較

In [ ]:
baseline = pd.DataFrame([{
    'Model': 'qwen2.5:1.5b before fine-tuning',
    'Accuracy': 0.5684,
    'Precision': 0.5416,
    'Recall': 0.8903,
    'F1-score': 0.6735,
    'TP': 560,
    'TN': 155,
    'FP': 474,
    'FN': 69,
}])

after = overall.copy()
after.insert(0, 'Model', 'Qwen2.5-1.5B-Instruct after LoRA BF16 fine-tuning')
after = after.rename(columns={'TP': 'TP', 'TN': 'TN', 'FP': 'FP', 'FN': 'FN'})
after = after[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'TP', 'TN', 'FP', 'FN']]

comparison = pd.concat([baseline, after], ignore_index=True)
display(comparison)
comparison.to_csv(f'{OUTPUT_DIR}/before_after_comparison.csv', index=False, encoding='utf-8-sig')